# Usage Forecast with LightGBM
## Using Snowflake Phone Usage Data - 3 Month Forecast

This notebook trains a usage forecasting model using:
- **Data Source**: Snowflake tables (PHONE_USAGE_DATA, ACCOUNT_ATTRIBUTES_MONTHLY)
- **Model**: LightGBM Regressor
- **Features**: Usage metrics with rolling windows, difference features, tenure, and ARR
- **Target**: Predict total phone calls (PHONE_TOTAL_CALLS) for the next 3 months

**Forecast Target**: Predict the total number of phone calls a customer will make 3 months in the future based on historical usage patterns and account characteristics.

---

## 1. Setup and Imports

In [ ]:
# Core libraries
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
import pickle
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# LightGBM
import lightgbm as lgb

# Sklearn
from sklearn import metrics
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)

# Snowflake
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, lit, count, sum as spark_sum

# Progress bar
from tqdm import tqdm

print("✓ Libraries imported successfully")
print(f"✓ LightGBM version: {lgb.__version__}")

## 2. Connect to Snowflake

In [ ]:
# Get active Snowflake session
session = get_active_session()

# Set database and schema
session.use_database("MY_DATABASE")
session.use_schema("PUBLIC")

print("✓ Snowflake session active")
print(f"  Database: {session.get_current_database()}")
print(f"  Schema: {session.get_current_schema()}")
print(f"  Warehouse: {session.get_current_warehouse()}")
print(f"  Role: {session.get_current_role()}")

# Verify tables exist
print("\n✓ Verifying tables exist...")
tables_to_check = ["PHONE_USAGE_DATA", "ACCOUNT_ATTRIBUTES_MONTHLY"]
for table in tables_to_check:
    count = session.table(f"MY_DATABASE.PUBLIC.{table}").count()
    print(f"  {table}: {count:,} rows")

## 3. Load Data from Snowflake

In [ ]:
# Load data from Snowflake
print("Loading data from MY_DATABASE.PUBLIC...")

# 1. Usage data
usage_df = session.table("MY_DATABASE.PUBLIC.PHONE_USAGE_DATA").to_pandas()
usage_df['MONTH'] = pd.to_datetime(usage_df['MONTH'])
print(f"✓ PHONE_USAGE_DATA: {len(usage_df):,} rows")

# 2. Account attributes
account_df = session.table("MY_DATABASE.PUBLIC.ACCOUNT_ATTRIBUTES_MONTHLY").to_pandas()
account_df['MONTH'] = pd.to_datetime(account_df['MONTH'])
print(f"✓ ACCOUNT_ATTRIBUTES_MONTHLY: {len(account_df):,} rows")

print(f"\nData date range: {usage_df['MONTH'].min()} to {usage_df['MONTH'].max()}")
print(f"Unique accounts: {usage_df['USERID'].nunique():,}")

# Display schema
print("\n=== PHONE_USAGE_DATA Schema ===")
print(usage_df.dtypes)

print("\n=== Sample Usage Data ===")
print(usage_df.head())

## 4. Feature Engineering & Usage Forecast Target Creation

**Forecast Target**:
- Predict PHONE_TOTAL_CALLS 3 months in the future
- Based on historical usage patterns and account characteristics

In [ ]:
# Merge usage data with account attributes
df = usage_df.merge(
    account_df[['ENTERPRISE_ACCOUNT_ID', 'MONTH', 'PACKAGE_ID', 'TIER_ID']],
    left_on=['USERID', 'MONTH'],
    right_on=['ENTERPRISE_ACCOUNT_ID', 'MONTH'],
    how='left'
).drop('ENTERPRISE_ACCOUNT_ID', axis=1)

# Sort data by account and month
df = df.sort_values(['USERID', 'MONTH']).reset_index(drop=True)

print("\n" + "="*70)
print("FEATURE ENGINEERING")
print("="*70)

# Configuration
PREDICTION_WINDOW = 3  # Predict 3 months ahead

print(f"\nParameters:")
print(f"  Prediction window: {PREDICTION_WINDOW} months")
print(f"  Target: PHONE_TOTAL_CALLS")

# ============================================================
# 1. TENURE AND ARR FEATURES
# ============================================================
print("\n📊 Creating tenure and ARR features...")

# Calculate signup date (first month seen for each account)
account_signup = df.groupby('USERID')['MONTH'].min().reset_index()
account_signup.columns = ['USERID', 'signup_date']
df = df.merge(account_signup, on='USERID', how='left')

# Calculate tenure in months
df['tenure_months'] = ((df['MONTH'] - df['signup_date']).dt.days / 30.44).round().astype(int)

# Create ARR based on package tier
package_arr_map = {
    100: 12000,   # Small Business
    200: 36000,   # Medium Business
    300: 84000,   # Large Business
    400: 180000,  # Enterprise
    500: 300000   # Enterprise Plus
}

df['ARR'] = df['PACKAGE_ID'].map(package_arr_map).fillna(36000)
df['ARR'] = df['ARR'] * np.random.uniform(0.9, 1.1, size=len(df))

# Calculate ARR changes
df['ARR_lag3'] = df.groupby('USERID')['ARR'].shift(3)
df['ARR_lag6'] = df.groupby('USERID')['ARR'].shift(6)
df['ARR_change_3m'] = df['ARR'] - df['ARR_lag3']
df['ARR_change_6m'] = df['ARR'] - df['ARR_lag6']
df['ARR_change_pct_3m'] = (df['ARR_change_3m'] / (df['ARR_lag3'] + 1)) * 100
df['ARR_change_pct_6m'] = (df['ARR_change_6m'] / (df['ARR_lag6'] + 1)) * 100

print(f"  ✓ Tenure and ARR features created")

# ============================================================
# 2. LAG FEATURES (3-month and 6-month lags)
# ============================================================
print("\n📊 Creating lag features...")

for col_name in ['PHONE_TOTAL_CALLS', 'PHONE_TOTAL_MINUTES_OF_USE', 'PHONE_MAU']:
    df[f'{col_name}_lag3'] = df.groupby('USERID')[col_name].shift(3)
    df[f'{col_name}_lag6'] = df.groupby('USERID')[col_name].shift(6)

# ============================================================
# 3. DIFFERENCE FEATURES
# ============================================================
print("📊 Creating difference features...")

for col_name in ['PHONE_TOTAL_CALLS', 'PHONE_TOTAL_MINUTES_OF_USE', 'PHONE_MAU']:
    df[f'{col_name}_diff3'] = df[col_name] - df[f'{col_name}_lag3']
    df[f'{col_name}_diff6'] = df[col_name] - df[f'{col_name}_lag6']

# ============================================================
# 4. RATIO FEATURES
# ============================================================
print("📊 Creating ratio features...")

df['voice_calls_ratio'] = df['VOICE_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['fax_calls_ratio'] = df['FAX_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['inbound_calls_ratio'] = df['PHONE_TOTAL_NUM_INBOUND_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['outbound_calls_ratio'] = df['PHONE_TOTAL_NUM_OUTBOUND_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['hardphone_ratio'] = df['HARDPHONE_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['softphone_ratio'] = df['SOFTPHONE_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['mobile_ratio'] = df['MOBILE_CALLS'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['minutes_per_call'] = df['PHONE_TOTAL_MINUTES_OF_USE'] / (df['PHONE_TOTAL_CALLS'] + 1)
df['mau_to_calls_ratio'] = df['PHONE_MAU'] / (df['PHONE_TOTAL_CALLS'] + 1)

# ============================================================
# 5. ROLLING FEATURES
# ============================================================
print("📊 Creating rolling aggregates...")

for col_name in ['PHONE_TOTAL_CALLS', 'PHONE_TOTAL_MINUTES_OF_USE', 'VOICE_CALLS']:
    df[f'{col_name}_roll3_mean'] = df.groupby('USERID')[col_name].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    df[f'{col_name}_roll6_mean'] = df.groupby('USERID')[col_name].transform(
        lambda x: x.rolling(window=6, min_periods=1).mean()
    )

# ============================================================
# 6. CREATE USAGE FORECAST TARGET
# ============================================================
print("\n" + "="*70)
print("CREATING USAGE FORECAST TARGET")
print("="*70)

# Get future usage (3 months ahead) - this is our target
df['future_usage'] = df.groupby('USERID')['PHONE_TOTAL_CALLS'].shift(-PREDICTION_WINDOW)

# Calculate current usage statistics for context
df['current_usage_avg'] = df.groupby('USERID')['PHONE_TOTAL_CALLS'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

print(f"\n✓ Usage forecast target created")
print(f"  Target: PHONE_TOTAL_CALLS {PREDICTION_WINDOW} months ahead")
print(f"  Rows with future usage data: {df['future_usage'].notna().sum():,}")
print(f"  Average future usage: {df['future_usage'].mean():.2f}")
print(f"  Min future usage: {df['future_usage'].min():.2f}")
print(f"  Max future usage: {df['future_usage'].max():.2f}")

# Remove rows where we can't calculate target or features
df_clean = df[
    df['future_usage'].notna() &
    df['PHONE_TOTAL_CALLS_lag3'].notna()
].copy()

# Rename target column for clarity
df_clean['usage_target'] = df_clean['future_usage']

print(f"\n✓ After filtering:")
print(f"  Rows with valid target: {len(df_clean):,}")
print(f"  Average target usage: {df_clean['usage_target'].mean():.2f}")
print(f"  Target std: {df_clean['usage_target'].std():.2f}")

## 5. Define Feature Sets

In [ ]:
# Define predictors
print("\n" + "="*70)
print("DEFINING FEATURE SETS")
print("="*70)

# Base usage features
base_features = [
    'PHONE_TOTAL_CALLS',
    'PHONE_TOTAL_MINUTES_OF_USE',
    'VOICE_CALLS',
    'FAX_CALLS',
    'PHONE_TOTAL_NUM_INBOUND_CALLS',
    'PHONE_TOTAL_NUM_OUTBOUND_CALLS',
    'HARDPHONE_CALLS',
    'SOFTPHONE_CALLS',
    'MOBILE_CALLS',
    'PHONE_MAU'
]

# Account features
account_features = [
    'tenure_months',
    'ARR',
    'PACKAGE_ID',
    'TIER_ID'
]

# ARR change features
arr_change_features = [
    'ARR_change_3m',
    'ARR_change_6m',
    'ARR_change_pct_3m',
    'ARR_change_pct_6m'
]

# Lag features
lag_features = [
    'PHONE_TOTAL_CALLS_lag3',
    'PHONE_TOTAL_CALLS_lag6',
    'PHONE_TOTAL_MINUTES_OF_USE_lag3',
    'PHONE_TOTAL_MINUTES_OF_USE_lag6',
    'PHONE_MAU_lag3',
    'PHONE_MAU_lag6'
]

# Difference features
diff_features = [
    'PHONE_TOTAL_CALLS_diff3',
    'PHONE_TOTAL_CALLS_diff6',
    'PHONE_TOTAL_MINUTES_OF_USE_diff3',
    'PHONE_TOTAL_MINUTES_OF_USE_diff6',
    'PHONE_MAU_diff3',
    'PHONE_MAU_diff6'
]

# Ratio features
ratio_features = [
    'voice_calls_ratio',
    'fax_calls_ratio',
    'inbound_calls_ratio',
    'outbound_calls_ratio',
    'hardphone_ratio',
    'softphone_ratio',
    'mobile_ratio',
    'minutes_per_call',
    'mau_to_calls_ratio'
]

# Rolling features
rolling_features = [
    'PHONE_TOTAL_CALLS_roll3_mean',
    'PHONE_TOTAL_CALLS_roll6_mean',
    'PHONE_TOTAL_MINUTES_OF_USE_roll3_mean',
    'PHONE_TOTAL_MINUTES_OF_USE_roll6_mean',
    'VOICE_CALLS_roll3_mean',
    'VOICE_CALLS_roll6_mean'
]

# Combine all features
predictors = (base_features + account_features + arr_change_features + 
              lag_features + diff_features + ratio_features + rolling_features)

print(f"\n📊 Feature Groups:")
print(f"  Base features: {len(base_features)}")
print(f"  Account features: {len(account_features)}")
print(f"  ARR change features: {len(arr_change_features)}")
print(f"  Lag features: {len(lag_features)}")
print(f"  Difference features: {len(diff_features)}")
print(f"  Ratio features: {len(ratio_features)}")
print(f"  Rolling features: {len(rolling_features)}")
print(f"  Total predictors: {len(predictors)}")

# Handle missing values
missing_counts = df_clean[predictors].isnull().sum()
if missing_counts.sum() > 0:
    print(f"\n⚠️ Features with missing values:")
    print(missing_counts[missing_counts > 0])
    print(f"\n  Filling missing values with 0...")
    df_clean[predictors] = df_clean[predictors].fillna(0)
else:
    print(f"\n✓ No missing values in predictors")

## 6. Split Data (Time-based Split)

In [ ]:
print("\n" + "="*70)
print("DATA SPLIT (Time-based)")
print("="*70)

# Sort by month
df_clean = df_clean.sort_values('MONTH').reset_index(drop=True)

# Get unique months
unique_months = sorted(df_clean['MONTH'].unique())
print(f"\n📅 Data range: {unique_months[0].strftime('%Y-%m')} to {unique_months[-1].strftime('%Y-%m')}")
print(f"   Total months: {len(unique_months)}")

# Time-based split
train_cutoff = pd.to_datetime('2024-11-01')
test_start = pd.to_datetime('2025-03-01')

train_df = df_clean[df_clean['MONTH'] < train_cutoff].copy()
test_df = df_clean[df_clean['MONTH'] >= test_start].copy()

# Validation set
val_size = int(len(train_df) * 0.2)
val_df = train_df.tail(val_size).copy()
train_df = train_df.head(len(train_df) - val_size).copy()

print(f"\n📊 Data Split:")
print(f"  Train: {len(train_df):,} samples")
print(f"    Avg target usage: {train_df['usage_target'].mean():.2f}")

print(f"\n  Validation: {len(val_df):,} samples")
print(f"    Avg target usage: {val_df['usage_target'].mean():.2f}")

print(f"\n  Test: {len(test_df):,} samples")
print(f"    Avg target usage: {test_df['usage_target'].mean():.2f}")

# Extract X and y
X_train = train_df[predictors]
y_train = train_df['usage_target']

X_val = val_df[predictors]
y_val = val_df['usage_target']

X_test = test_df[predictors]
y_test = test_df['usage_target']

print(f"\n✓ Feature matrices created")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_val shape: {X_val.shape}")
print(f"  X_test shape: {X_test.shape}")

## 7. Train LightGBM Model

In [ ]:
print("\n" + "="*70)
print("TRAINING LIGHTGBM MODEL")
print("="*70)

# Model parameters
model_params = {
    "num_iterations": 100,
    "learning_rate": 0.1,
    "max_depth": -1,
    "min_data_in_leaf": 100,
    "metric": "rmse",
    "random_state": 42,
    "verbose": -1
}

print(f"\n📋 Model Parameters:")
for param, value in model_params.items():
    print(f"  {param}: {value}")

# Initialize model
model = lgb.LGBMRegressor(**model_params)

# Train model
print(f"\n🚀 Training model...")

model.fit(
    X_train, 
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.log_evaluation(period=10)
    ]
)

print(f"\n✓ Model training completed!")

## 8. Feature Importance

In [ ]:
# Get feature importance
importance = model.feature_importances_
feat_importance = pd.DataFrame({
    "feature": predictors,
    "importance": importance
}).sort_values("importance", ascending=False)

print("\n" + "="*70)
print("TOP 20 FEATURE IMPORTANCES")
print("="*70)
print(feat_importance.head(20).to_string(index=False))

# Plot feature importance
top_n = 20
plt.figure(figsize=(10, 8))
plt.barh(
    feat_importance["feature"].iloc[:top_n][::-1],
    feat_importance["importance"].iloc[:top_n][::-1],
    color='coral'
)
plt.xlabel('Importance', fontsize=10)
plt.ylabel('Feature', fontsize=10)
plt.title(f"Top {top_n} Feature Importances - Usage Forecast", fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print(f"\n✓ Feature importance plotted")

## 9. Model Evaluation

In [ ]:
print("\n" + "="*70)
print("MODEL EVALUATION")
print("="*70)

# Predictions
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

# Evaluate function
def evaluate_predictions(y_true, y_pred, dataset_name="Dataset"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # MAPE (handle division by zero)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-6))) * 100
    
    print(f"\n📊 {dataset_name} Results:")
    print(f"  MAE:  {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAPE: {mape:.2f}%")
    
    return mae, rmse, r2, mape

# Evaluate all sets
train_mae, train_rmse, train_r2, train_mape = evaluate_predictions(
    y_train, y_train_pred, dataset_name="Training Set"
)

val_mae, val_rmse, val_r2, val_mape = evaluate_predictions(
    y_val, y_val_pred, dataset_name="Validation Set"
)

test_mae, test_rmse, test_r2, test_mape = evaluate_predictions(
    y_test, y_test_pred, dataset_name="Test Set"
)

In [ ]:
# Prediction Statistics
print("\n" + "="*70)
print("PREDICTION STATISTICS (Test Set)")
print("="*70)
print(f"\nActual Usage Statistics:")
print(f"  Mean:   {y_test.mean():.2f}")
print(f"  Median: {y_test.median():.2f}")
print(f"  Std:    {y_test.std():.2f}")
print(f"  Min:    {y_test.min():.2f}")
print(f"  Max:    {y_test.max():.2f}")

print(f"\nPredicted Usage Statistics:")
print(f"  Mean:   {y_test_pred.mean():.2f}")
print(f"  Median: {np.median(y_test_pred):.2f}")
print(f"  Std:    {y_test_pred.std():.2f}")
print(f"  Min:    {y_test_pred.min():.2f}")
print(f"  Max:    {y_test_pred.max():.2f}")

# Error distribution
errors = y_test - y_test_pred
print(f"\nError Statistics:")
print(f"  Mean Error:   {errors.mean():.2f}")
print(f"  Std Error:    {errors.std():.2f}")
print(f"  Min Error:    {errors.min():.2f}")
print(f"  Max Error:    {errors.max():.2f}")

## 10. Visualizations

In [ ]:
# Scatter plots: Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.3, s=10)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Usage', fontsize=10)
axes[0].set_ylabel('Predicted Usage', fontsize=10)
axes[0].set_title(f'Training Set (R² = {train_r2:.3f})', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.3, s=10, color='red')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Usage', fontsize=10)
axes[1].set_ylabel('Predicted Usage', fontsize=10)
axes[1].set_title(f'Test Set (R² = {test_r2:.3f})', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Scatter plots (Predicted vs Actual) plotted")

In [ ]:
# Residual Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
residuals = y_test - y_test_pred
axes[0].scatter(y_test_pred, residuals, alpha=0.3, s=10, color='blue')
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Usage', fontsize=10)
axes[0].set_ylabel('Residuals (Actual - Predicted)', fontsize=10)
axes[0].set_title('Residual Plot - Test Set', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Error Distribution
axes[1].hist(residuals, bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero Error')
axes[1].set_xlabel('Residuals', fontsize=10)
axes[1].set_ylabel('Frequency', fontsize=10)
axes[1].set_title('Error Distribution - Test Set', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✓ Residual plots plotted")

## 11. Generate Predictions

In [ ]:
# Create predictions dataframe
predictions_df = test_df[['USERID', 'MONTH']].copy()
predictions_df['predicted_usage'] = y_test_pred
predictions_df['actual_usage'] = y_test.values
predictions_df['prediction_error'] = predictions_df['actual_usage'] - predictions_df['predicted_usage']
predictions_df['absolute_error'] = np.abs(predictions_df['prediction_error'])
predictions_df['percentage_error'] = (predictions_df['prediction_error'] / (predictions_df['actual_usage'] + 1e-6)) * 100

print("\n" + "="*70)
print("PREDICTIONS GENERATED")
print("="*70)
print(f"\n✓ Generated predictions for {len(predictions_df):,} test samples")
print(f"\nSample predictions:")
print(predictions_df.head(10))

# Accounts with largest prediction errors
print(f"\n=== Largest Over-predictions (Predicted >> Actual) ===")
over_pred = predictions_df.nlargest(10, 'prediction_error')[['USERID', 'MONTH', 'predicted_usage', 'actual_usage', 'prediction_error']]
print(over_pred)

print(f"\n=== Largest Under-predictions (Predicted << Actual) ===")
under_pred = predictions_df.nsmallest(10, 'prediction_error')[['USERID', 'MONTH', 'predicted_usage', 'actual_usage', 'prediction_error']]
print(under_pred)

## 12. Save Results to Snowflake

In [ ]:
# Save predictions
print("\n" + "="*70)
print("SAVING TO SNOWFLAKE")
print("="*70)

print("\n📤 Saving predictions to MY_DATABASE.PUBLIC.USAGE_FORECAST_LGBM...")
try:
    predictions_snowpark = session.create_dataframe(predictions_df)
    predictions_snowpark.write.mode("overwrite").save_as_table("MY_DATABASE.PUBLIC.USAGE_FORECAST_LGBM")
    
    result_count = session.table("MY_DATABASE.PUBLIC.USAGE_FORECAST_LGBM").count()
    print(f"✓ Saved {result_count:,} predictions to USAGE_FORECAST_LGBM table")
except Exception as e:
    print(f"✗ Error saving predictions: {str(e)}")

In [ ]:
# Save model metrics
print("\n📤 Saving model metrics to MY_DATABASE.PUBLIC.USAGE_FORECAST_METRICS_LGBM...")

try:
    metrics_df = pd.DataFrame({
        'model_name': ['LightGBM_UsageForecast'],
        'train_date': [datetime.now()],
        'test_mae': [test_mae],
        'test_rmse': [test_rmse],
        'test_r2': [test_r2],
        'test_mape': [test_mape],
        'num_features': [len(predictors)],
        'num_iterations': [model_params['num_iterations']],
        'learning_rate': [model_params['learning_rate']],
        'train_samples': [len(train_df)],
        'test_samples': [len(test_df)],
        'prediction_window_months': [PREDICTION_WINDOW],
        'target_variable': ['PHONE_TOTAL_CALLS']
    })
    
    metrics_snowpark = session.create_dataframe(metrics_df)
    metrics_snowpark.write.mode("append").save_as_table("MY_DATABASE.PUBLIC.USAGE_FORECAST_METRICS_LGBM")
    
    print(f"✓ Saved model metrics to USAGE_FORECAST_METRICS_LGBM table")
except Exception as e:
    print(f"✗ Error saving metrics: {str(e)}")

## 13. Save Model Files

In [ ]:
print("\n" + "="*70)
print("SAVING MODEL FILES")
print("="*70)

# Create model package
model_package = {
    'version': 'v1',
    'model': model,
    'predictors': predictors,
    'model_params': model_params,
    'test_metrics': {
        'mae': test_mae,
        'rmse': test_rmse,
        'r2': test_r2,
        'mape': test_mape
    },
    'metadata': {
        'version': 'v1',
        'description': 'LightGBM model for usage forecast (3-month forecast)',
        'train_date': datetime.now().isoformat(),
        'num_features': len(predictors),
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'prediction_window_months': PREDICTION_WINDOW,
        'target_variable': 'PHONE_TOTAL_CALLS',
        'model_type': 'regression'
    }
}

# Save locally
with open('usage_forecast_lgbm_v1.pkl', 'wb') as f:
    pickle.dump(model_package, f)
print("✓ Saved: usage_forecast_lgbm_v1.pkl")

# Upload to Snowflake
print("\n📤 Uploading to Snowflake...")
try:
    session.sql("CREATE STAGE IF NOT EXISTS MY_DATABASE.PUBLIC.MODELS").collect()
    
    session.file.put(
        'usage_forecast_lgbm_v1.pkl',
        '@MY_DATABASE.PUBLIC.MODELS/usage_forecast/v1/',
        auto_compress=False,
        overwrite=True
    )
    print("✓ Uploaded to: @MY_DATABASE.PUBLIC.MODELS/usage_forecast/v1/usage_forecast_lgbm_v1.pkl")
    
    # Register in model registry
    session.sql("""
        CREATE TABLE IF NOT EXISTS MY_DATABASE.PUBLIC.USAGE_FORECAST_MODEL_REGISTRY_LGBM (
            VERSION VARCHAR(50) PRIMARY KEY,
            MODEL_PATH VARCHAR(500),
            MODEL_TYPE VARCHAR(100),
            DESCRIPTION VARCHAR(1000),
            MAE FLOAT,
            RMSE FLOAT,
            R2 FLOAT,
            MAPE FLOAT,
            TRAIN_DATE TIMESTAMP,
            IS_PRODUCTION BOOLEAN DEFAULT FALSE,
            PREDICTION_WINDOW_MONTHS INT,
            TARGET_VARIABLE VARCHAR(100)
        )
    """).collect()
    
    session.sql(f"""
        MERGE INTO MY_DATABASE.PUBLIC.USAGE_FORECAST_MODEL_REGISTRY_LGBM AS target
        USING (SELECT 'v1' AS VERSION) AS source
        ON target.VERSION = source.VERSION
        WHEN MATCHED THEN UPDATE SET
            MODEL_PATH = '@MY_DATABASE.PUBLIC.MODELS/usage_forecast/v1/usage_forecast_lgbm_v1.pkl',
            MODEL_TYPE = 'LightGBM',
            DESCRIPTION = 'LightGBM usage forecast model - 3 month forecast',
            MAE = {test_mae},
            RMSE = {test_rmse},
            R2 = {test_r2},
            MAPE = {test_mape},
            TRAIN_DATE = CURRENT_TIMESTAMP(),
            PREDICTION_WINDOW_MONTHS = {PREDICTION_WINDOW},
            TARGET_VARIABLE = 'PHONE_TOTAL_CALLS'
        WHEN NOT MATCHED THEN INSERT (
            VERSION, MODEL_PATH, MODEL_TYPE, DESCRIPTION, MAE, RMSE, R2, MAPE,
            TRAIN_DATE, PREDICTION_WINDOW_MONTHS, TARGET_VARIABLE
        )
        VALUES (
            'v1', '@MY_DATABASE.PUBLIC.MODELS/usage_forecast/v1/usage_forecast_lgbm_v1.pkl', 'LightGBM',
            'LightGBM usage forecast model - 3 month forecast',
            {test_mae}, {test_rmse}, {test_r2}, {test_mape},
            CURRENT_TIMESTAMP(), {PREDICTION_WINDOW}, 'PHONE_TOTAL_CALLS'
        )
    """).collect()
    
    print("✓ Registered in USAGE_FORECAST_MODEL_REGISTRY_LGBM")
    
except Exception as e:
    print(f"⚠️ Error uploading to Snowflake: {e}")

print("\n" + "="*70)
print("✓ MODEL SAVED SUCCESSFULLY")
print("="*70)
print(f"\n📊 Model: LightGBM Usage Forecast Regressor v1")
print(f"📁 Local file: usage_forecast_lgbm_v1.pkl")
print(f"☁️  Snowflake: @MY_DATABASE.PUBLIC.MODELS/usage_forecast/v1/")
print(f"\n📈 Performance:")
print(f"  MAE:  {test_mae:.2f}")
print(f"  RMSE: {test_rmse:.2f}")
print(f"  R²:   {test_r2:.4f}")
print(f"  MAPE: {test_mape:.2f}%")
print(f"\n🎯 Configuration:")
print(f"  Features: {len(predictors)}")
print(f"  Prediction window: {PREDICTION_WINDOW} months")
print(f"  Target variable: PHONE_TOTAL_CALLS")
print(f"  Model type: Regression")

## 14. Summary

This notebook successfully:
1. ✅ Loaded phone usage data and account attributes from Snowflake
2. ✅ Created comprehensive features (lag, difference, ratio, rolling, tenure, ARR)
3. ✅ Defined usage forecast target (PHONE_TOTAL_CALLS 3 months ahead)
4. ✅ Trained LightGBM regressor
5. ✅ Evaluated model performance with regression metrics
6. ✅ Analyzed feature importance
7. ✅ Generated usage predictions
8. ✅ Saved results and model to Snowflake

### Key Insights:
- Uses LightGBM for gradient boosting regression
- Predicts future usage based on historical patterns
- 3-month forecast window for usage planning
- Includes tenure and ARR features for account context
- Uses regression metrics: MAE, RMSE, R², MAPE

### Next Steps:
- Fine-tune hyperparameters for better accuracy
- Experiment with different prediction windows
- Add seasonality features for better forecasts
- Monitor model performance over time
- Use predictions for capacity planning and resource allocation